# Lean merged.csv performance and ratio plots

Self-contained plotting notebook for `merged.csv`-style result tables. The main plot copies the two-row `plot_mutation_model_performance` structure: boxplots by test-mutation regime on top, and performance across train mutations on bottom.


In [ ]:
from pathlib import Path
from typing import Literal
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

plt.rcParams["svg.fonttype"] = "none"

DATA_PATH = Path("~/Desktop/plm_paper/merged.csv").expanduser()
OUT_DIR = Path("~/Desktop/plm_paper/refined_figures").expanduser()
SAVE_FIGURES = True

DEFAULT_FEATURES = ["roc", "top_100_pct", "correlation"]
PRED_BEGIN = 1
MIN_MUTS = 3
N_MUTS_TO_CHECK = 6

FEATURE_LABELS = {
    "roc": "ROC-AUC",
    "top_100_pct": "Top 100%",
    "correlation": "Correlation",
}

DATASET_LABELS = {
    "gfp": "GFP",
    "pard3": "PARD3",
    "nmt": "NMT",
    "gcn": "GCN4",
    "gcn4": "GCN4",
    "lov": "LOV",
    "his": "HIS",
    "his2": "HIS2",
    "his5": "HIS5",
    "casp": "CASP",
}

PLM_COLOR_MAP = {
    "esm_35m": "#9adffc",
    "esm_8m": "#4a95ff",
    "esm_150m": "#131791",
    "esm_650m": "#00a087",
    "esm_3b": "#f0b800",
    "progen2-small": "#fb8072",
    "progen2-medium": "#b15928",
    "prot_bert": "#984ea3",
}

LLM_OHE_COLOR_MAP = {
    "llm": "#f0b800",
    "one_hot": "#5d0187",
}

CLF_COLOR_MAP = {
    "xgboost": "#9adffc",
    "ridgeregression": "#4a95ff",
    "mlp": "#131791",
    "linreg": "#5d0187",
}


## Load data

In [ ]:
df_all = pd.read_csv(DATA_PATH)
print(df_all.shape)
display(df_all.head())


## Shared helpers

In [ ]:
def display_dataset(value):
    return DATASET_LABELS.get(str(value), str(value).upper())


def display_feature(value):
    return FEATURE_LABELS.get(str(value), str(value).replace("_", " ").title())


def available_features(df, features=DEFAULT_FEATURES):
    return [feature for feature in features if feature in df.columns and df[feature].notna().any()]


def grouping_key(df, grouping_columns):
    if isinstance(grouping_columns, str):
        grouping_columns = [grouping_columns]
    if len(grouping_columns) == 1:
        return df[grouping_columns[0]].astype(str)
    return df[grouping_columns].astype(str).agg("_".join, axis=1)


def style_axes(ax):
    ax.grid(True, which="both", linestyle="--", linewidth=0.25, alpha=0.7)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)


def save_fig(fig, path):
    if SAVE_FIGURES and path is not None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, bbox_inches="tight")


## Mutation heatmaps

In [ ]:
def plot_mutation_heatmaps_by_dataset(
    df: pd.DataFrame,
    mode: Literal["train", "test"],
    metric: Literal["roc", "correlation"],
    *,
    datasets: list[str] | None = None,
    model_column: str = "model_name",
    aggfunc: str = "mean",
    subplot_width: float = 5,
    height: float = 6,
    cmap: str | None = None,
    annotate: bool = True,
    share_color_scale: bool = True,
    ncols: int | None = None,
    layout: tuple[int, int] | None = None,
    vmin: float | None = None,
    vmax: float | None = None,
    center: float | None = None,
    output_path=None,
) -> tuple[plt.Figure, list[plt.Axes]]:
    """
    Plot mutation heatmaps by dataset.

    Each subplot represents one dataset:
      - Rows: model_column
      - Columns: train_mutations or test_mutations
      - Cell value: roc or correlation
    """
    mutation_column = {"train": "train_mutations", "test": "test_mutations"}[mode]
    required_columns = {"dataset", model_column, mutation_column, metric}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    plot_df = df.dropna(subset=["dataset", model_column, mutation_column, metric]).copy()
    if datasets is not None:
        plot_df = plot_df[plot_df["dataset"].isin(datasets)]
        dataset_names = [dataset for dataset in datasets if dataset in set(plot_df["dataset"])]
    else:
        dataset_names = plot_df["dataset"].drop_duplicates().tolist()

    if not dataset_names:
        raise ValueError("No valid data is available to plot.")

    if metric == "correlation":
        heatmap_options = {"cmap": cmap or "vlag", "center": 0 if center is None else center}
        if share_color_scale:
            heatmap_options.update({"vmin": -1 if vmin is None else vmin, "vmax": 1 if vmax is None else vmax})
    else:
        heatmap_options = {"cmap": cmap or "viridis"}
        if share_color_scale:
            heatmap_options.update({"vmin": 0 if vmin is None else vmin, "vmax": 1 if vmax is None else vmax})
        elif center is not None:
            heatmap_options["center"] = center

    nplots = len(dataset_names)
    if layout is not None:
        nrows, ncols = layout
    else:
        ncols = nplots if ncols is None else ncols
        nrows = int(np.ceil(nplots / ncols))
    if nrows * ncols < nplots:
        raise ValueError("layout is too small for the selected datasets")

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(subplot_width * ncols, height * nrows),
        squeeze=False,
    )
    axes = list(axes.flatten())

    for index, (ax, dataset_name) in enumerate(zip(axes, dataset_names)):
        dataset_df = plot_df.loc[plot_df["dataset"] == dataset_name]
        heatmap_data = dataset_df.pivot_table(
            index=model_column,
            columns=mutation_column,
            values=metric,
            aggfunc=aggfunc,
            sort=False,
        )

        sns.heatmap(
            heatmap_data,
            ax=ax,
            annot=annotate,
            fmt=".2f",
            linewidths=0.5,
            cbar=(index == nplots - 1) if share_color_scale else True,
            cbar_kws={"label": metric},
            **heatmap_options,
        )
        ax.set_title(str(display_dataset(dataset_name)))
        ax.set_xlabel(mutation_column.replace("_", " ").title())
        ax.set_ylabel("Model" if index % ncols == 0 else "")
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    for ax in axes[nplots:]:
        fig.delaxes(ax)

    fig.suptitle(f"{metric.replace('_', ' ').title()} by {mode.title()} Mutation", fontsize=14)
    fig.tight_layout()
    save_fig(fig, output_path)
    return fig, axes[:nplots]


# Examples: choose datasets and layout explicitly.
plot_mutation_heatmaps_by_dataset(
    df_all[(df_all["clf_type"] == "mlp") & (~df_all["model_name"].isin(["one_hot", "linreg"]))],
    mode="train",
    metric="roc",
    datasets=["gfp", "his2", "his5"],
    ncols=3,
    height=4,
    annotate=True,
    share_color_scale=True,
    output_path=OUT_DIR / "heatmaps" / "roc_by_train_mutations.svg",
)
plt.show()

plot_mutation_heatmaps_by_dataset(
    df_all[(df_all["clf_type"] == "mlp") & (~df_all["model_name"].isin(["one_hot", "linreg"]))],
    mode="test",
    metric="correlation",
    datasets=["casp", "gcn4", "lov", "nmt", "pard3"],
    ncols=3,
    height=4,
    annotate=True,
    share_color_scale=True,
    output_path=OUT_DIR / "heatmaps" / "correlation_by_test_mutations.svg",
)
plt.show()


## Mutation-performance plot

`grouping_columns` can be a string or a list. For multiple grouping columns, group names are joined with `_`, so your `color_map` should use keys like `a_b`.


In [ ]:
def plot_mutation_model_performance(
    df,
    *,
    grouping_columns,
    color_map,
    feature_to_plot,
    pred_begin=PRED_BEGIN,
    min_muts=MIN_MUTS,
    n_muts_to_check=N_MUTS_TO_CHECK,
    output_path=None,
    min_max_y=True,
):
    plot_df = df.copy()
    plot_df["group_key"] = grouping_key(plot_df, grouping_columns)
    plot_df = plot_df[plot_df["group_key"].isin(color_map.keys())]

    col_size = 1.35
    fig, axs = plt.subplots(
        2,
        n_muts_to_check,
        figsize=(col_size * n_muts_to_check, 0.3 + col_size * 2),
        constrained_layout=True,
    )
    if n_muts_to_check == 1:
        axs = np.asarray(axs).reshape(2, 1)

    global_y = plot_df[feature_to_plot].dropna().astype(float)
    global_ymin, global_ymax = (global_y.min(), global_y.max()) if len(global_y) else (0, 1)
    y_pad = (global_ymax - global_ymin) * 0.1 if global_ymax > global_ymin else 0.05

    for test_mut in range(min_muts, min_muts + n_muts_to_check):
        col = test_mut - min_muts
        ax_box, ax_line = axs[0, col], axs[1, col]
        style_axes(ax_box)
        style_axes(ax_line)

        test_df = plot_df[plot_df["test_mutations"] == test_mut]
        train_df = test_df[(test_df["train_mutations"] < test_mut) & (test_df["train_mutations"] >= pred_begin)]

        boxplot_data = []
        group_names = []
        for group in color_map:
            vals = test_df.loc[test_df["group_key"] == group, feature_to_plot].dropna().astype(float).to_numpy()
            if len(vals):
                boxplot_data.append(vals)
                group_names.append(group)

        if boxplot_data:
            bp = ax_box.boxplot(
                boxplot_data,
                patch_artist=True,
                boxprops=dict(linewidth=1, color="black"),
                medianprops=dict(color="black", linewidth=1),
                whiskerprops=dict(linewidth=1, color="black"),
                capprops=dict(linewidth=1, color="black"),
                flierprops=dict(markerfacecolor="black", marker="o", markersize=3.5, markeredgecolor="black", linestyle="none"),
            )
            for patch, group in zip(bp["boxes"], group_names):
                patch.set_facecolor(color_map[group])

        ax_box.set_title(f"Mutations in test: {test_mut}\n", fontsize=8, pad=3)
        ax_box.set_xlabel("Model", fontsize=9, labelpad=3)
        ax_box.set_xticklabels([])
        if col == 0:
            ax_box.set_ylabel(display_feature(feature_to_plot), labelpad=6, fontsize=9)

        if min_max_y and len(global_y):
            ax_box.set_ylim(global_ymin - y_pad, global_ymax + y_pad)
            ax_line.set_ylim(global_ymin - y_pad, global_ymax + y_pad)
        if feature_to_plot == "top_100_pct":
            ax_box.set_ylim(0, 1.05)
            ax_line.set_ylim(0, 1.05)
            ax_box.set_yticks([0, 0.5, 1])
            ax_line.set_yticks([0, 0.5, 1])

        for group in color_map:
            sub = train_df[train_df["group_key"] == group]
            if sub.empty:
                continue
            summary = sub.groupby("train_mutations", as_index=False)[feature_to_plot].median().sort_values("train_mutations")
            ax_line.plot(
                summary["train_mutations"],
                summary[feature_to_plot],
                marker="o",
                color=color_map[group],
                label=group if col == 0 else None,
                markersize=3,
                linewidth=1,
                alpha=0.7,
            )

        ax_line.set_xlabel("Mutations\nin train", fontsize=9, labelpad=1)
        ax_line.tick_params(axis="x", labelsize=9)
        ax_line.tick_params(axis="y", labelsize=9)
        if col == 0:
            ax_line.set_ylabel(display_feature(feature_to_plot), labelpad=6, fontsize=9)
            handles, labels = ax_line.get_legend_handles_labels()
            if handles:
                ax_line.legend(loc="best", fontsize=7, frameon=False)

    save_fig(fig, output_path)
    return fig


## Example calls across datasets

In [ ]:
# PLM vs PLM. Assumes one dataset per call; this cell loops for convenience.
for dataset, dataset_df in df_all.groupby("dataset"):
    sub_df = dataset_df[(dataset_df["clf_type"] == "mlp") & (~dataset_df["model_name"].isin(["one_hot", "linreg"]))]
    for feature in available_features(sub_df):
        plot_mutation_model_performance(
            sub_df,
            grouping_columns="model_name",
            color_map=PLM_COLOR_MAP,
            feature_to_plot=feature,
            output_path=OUT_DIR / "figure_models_comparision" / str(dataset) / f"{feature}.svg",
            min_max_y=True,
        )
        plt.show()


In [ ]:
def best_plm_vs_one_hot_df(dataset_df, feature, *, plm_clf_type="mlp"):
    plm_df = dataset_df[(dataset_df["clf_type"] == plm_clf_type) & (~dataset_df["model_name"].isin(["one_hot", "linreg"]))].copy()
    ohe_df = dataset_df[dataset_df["model_name"].isin(["one_hot", "ohe"])].copy()
    if plm_df.empty or ohe_df.empty or feature not in dataset_df:
        return pd.DataFrame(), None

    best_model = plm_df.groupby("model_name")[feature].median().idxmax()
    best_df = plm_df[plm_df["model_name"] == best_model].copy()
    best_df["plot_group"] = "llm"
    ohe_df["plot_group"] = "one_hot"
    return pd.concat([best_df, ohe_df], ignore_index=True), best_model


# Best PLM vs one-hot.
for dataset, dataset_df in df_all.groupby("dataset"):
    for feature in available_features(dataset_df):
        plot_df, best_model = best_plm_vs_one_hot_df(dataset_df, feature)
        if plot_df.empty:
            continue
        print(dataset, feature, "best PLM:", best_model)
        plot_mutation_model_performance(
            plot_df,
            grouping_columns="plot_group",
            color_map=LLM_OHE_COLOR_MAP,
            feature_to_plot=feature,
            output_path=OUT_DIR / "figure_ohe_vs_llm" / str(dataset) / f"{feature}.svg",
            min_max_y=True,
        )
        plt.show()


In [ ]:
# Classifier type comparison within each dataset.
for dataset, dataset_df in df_all.groupby("dataset"):
    sub_df = dataset_df[dataset_df["clf_type"].isin(CLF_COLOR_MAP.keys())]
    for feature in available_features(sub_df):
        plot_mutation_model_performance(
            sub_df,
            grouping_columns="clf_type",
            color_map=CLF_COLOR_MAP,
            feature_to_plot=feature,
            output_path=OUT_DIR / "supp_figure_model_selection" / str(dataset) / f"{feature}.svg",
            min_max_y=True,
        )
        plt.show()


## Ratio calculation

In [ ]:
def calculate_best_plm_ohe_ratios(df, *, features=DEFAULT_FEATURES, plm_clf_type="mlp", epsilon=1e-8):
    rows = []
    for dataset, dataset_df in df.groupby("dataset"):
        for feature in available_features(dataset_df, features):
            plm_df = dataset_df[(dataset_df["clf_type"] == plm_clf_type) & (~dataset_df["model_name"].isin(["one_hot", "ohe", "linreg"]))]
            ohe_df = dataset_df[dataset_df["model_name"].isin(["one_hot", "ohe"])]
            if plm_df.empty or ohe_df.empty:
                continue

            best_model = plm_df.groupby("model_name")[feature].median().idxmax()
            best_df = plm_df[plm_df["model_name"] == best_model]

            llm = best_df.groupby(["train_mutations", "test_mutations"], as_index=False)[feature].median().rename(columns={feature: "llm_value"})
            ohe = ohe_df.groupby(["train_mutations", "test_mutations"], as_index=False)[feature].median().rename(columns={feature: "ohe_value"})
            merged = ohe.merge(llm, on=["train_mutations", "test_mutations"], how="inner")
            merged = merged[np.abs(merged["llm_value"]) > epsilon]

            for _, row in merged.iterrows():
                rows.append({
                    "dataset": dataset,
                    "feature": feature,
                    "title_for_plot": display_feature(feature),
                    "best_model": best_model,
                    "train_mutations": row["train_mutations"],
                    "test_mutations": row["test_mutations"],
                    "ratio": row["ohe_value"] / row["llm_value"],
                    "ohe_value": row["ohe_value"],
                    "llm_value": row["llm_value"],
                })
    return pd.DataFrame(rows)


ratio_df_all = calculate_best_plm_ohe_ratios(df_all)
print(ratio_df_all.shape)
display(ratio_df_all.head())


## Ratio summary plots

In [ ]:
def ratio_summary_table(ratio_df):
    plot_source = ratio_df.copy()
    plot_source["fancy_title"] = plot_source["title_for_plot"].fillna(plot_source["feature"])
    return (
        plot_source
        .groupby(["dataset", "fancy_title"])["ratio"]
        .agg(["mean", "sem"])
        .reset_index()
        .rename(columns={"mean": "mean_ratio", "sem": "sem_ratio"})
    )


def plot_ratio_summary(
    ratio_df,
    *,
    broken_axis=False,
    left_xlim=(0.5, 1.27),
    right_xlim=(1.9, 2.8),
    as_percent=False,
    annotate=True,
    output_path=None,
):
    summary_df = ratio_summary_table(ratio_df)
    if as_percent:
        summary_df["mean_ratio"] *= 100
        summary_df["sem_ratio"] *= 100

    hue_order = sorted(summary_df["dataset"].dropna().unique())
    y_order = list(summary_df["fancy_title"].drop_duplicates())
    mean_piv = summary_df.pivot(index="fancy_title", columns="dataset", values="mean_ratio").reindex(y_order)
    sem_piv = summary_df.pivot(index="fancy_title", columns="dataset", values="sem_ratio").reindex(y_order)

    if broken_axis:
        fig, axes = plt.subplots(
            1,
            2,
            sharey=True,
            figsize=(74.844 / 25.4, 137.214 / 25.4),
            gridspec_kw={"width_ratios": [left_xlim[1] - left_xlim[0], right_xlim[1] - right_xlim[0]], "wspace": 0.05},
        )
    else:
        fig, ax = plt.subplots(figsize=(3, 4))
        axes = (ax,)

    n_hue = len(hue_order)
    y = np.arange(len(y_order))
    bar_height = 0.8 / max(n_hue, 1)
    palette = sns.color_palette("tab10", n_colors=n_hue)

    for i, dataset in enumerate(hue_order):
        offset = (i - (n_hue - 1) / 2) * bar_height
        vals = mean_piv[dataset].to_numpy(dtype=float)
        errs = sem_piv[dataset].to_numpy(dtype=float)
        mask = ~np.isnan(vals)
        y_pos = y[mask] + offset
        vals_plot = vals[mask]
        errs_plot = errs[mask]

        for ax in axes:
            ax.barh(y_pos, vals_plot, height=bar_height, capsize=3, color=palette[i], label=dataset, edgecolor="none")
            if annotate:
                for yy, val, err in zip(y_pos, vals_plot, errs_plot):
                    if np.isnan(val):
                        continue
                    err = 0 if np.isnan(err) else err
                    text = f"{val:.2f}+/-{err:.2f}"
                    xloc = val + (0.02 if not as_percent else 2)
                    ax.text(xloc, yy, text, va="center", ha="left", fontsize=8, color="black")

    if broken_axis:
        ax1, ax2 = axes
        ax1.set_xlim(*left_xlim)
        ax2.set_xlim(*right_xlim)
        ax1.axvline(x=100 if as_percent else 1, color="k", linestyle="--", linewidth=1, alpha=0.7)
        ax2.tick_params(axis="y", left=False, labelleft=False)
        ax2.spines["left"].set_visible(False)
        ax2.spines["right"].set_visible(True)
        ax1.yaxis.tick_left()
        ax2.yaxis.tick_right()
        d = 0.015
        kwargs = dict(transform=ax1.transAxes, color="k", clip_on=False, linewidth=1)
        ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs)
        ax1.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)
        kwargs = dict(transform=ax2.transAxes, color="k", clip_on=False, linewidth=1)
        ax2.plot((-d, +d), (-d, +d), **kwargs)
        ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)
    else:
        axes[0].axvline(x=100 if as_percent else 1, color="k", linestyle="--", linewidth=1, alpha=0.7)
        axes[0].legend(title="Dataset", loc="best")

    axes[0].set_yticks(y)
    axes[0].set_yticklabels(y_order)
    axes[0].invert_yaxis()
    axes[0].set_ylabel("")
    axes[-1].set_xlabel("")
    axes[0].set_xlabel("Mean LLM/OHE Ratio (%)" if as_percent else "")

    for ax in axes:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False if ax is axes[0] else True)
        ax.xaxis.grid(True, which="both", linestyle="--", linewidth=0.6, alpha=0.7)
        ax.yaxis.grid(True, which="both", linestyle="--", linewidth=0.6, alpha=0.7)
        if ax.get_legend() is not None and broken_axis:
            ax.get_legend().remove()

    fig.tight_layout()
    save_fig(fig, output_path)
    return fig


In [ ]:
plot_ratio_summary(
    ratio_df_all,
    broken_axis=True,
    left_xlim=(0.5, 1.27),
    right_xlim=(1.9, 2.8),
    as_percent=False,
    annotate=True,
    output_path=OUT_DIR / "figure_ohe_vs_llm" / "improvement_boxplot_plot.svg",
)
plt.show()

plot_ratio_summary(
    ratio_df_all,
    broken_axis=False,
    as_percent=True,
    annotate=True,
    output_path=OUT_DIR / "figure_ohe_vs_llm" / "improvement_ratio_percent.svg",
)
plt.show()


## Ratio improvement distributions

In [ ]:
def resolve_ratio_plot_pairs(ratio_df, plot_pairs=None, datasets=None, features=None):
    if plot_pairs is None:
        datasets = ratio_df["dataset"].dropna().unique() if datasets is None else datasets
        features = ratio_df["feature"].dropna().unique() if features is None else features
        plot_pairs = [(dataset, feature) for dataset in datasets for feature in features]

    resolved = []
    for dataset, feature in plot_pairs:
        group = ratio_df[(ratio_df["dataset"] == dataset) & (ratio_df["feature"] == feature)]
        if group.empty:
            continue
        title = group["title_for_plot"].dropna().iloc[0] if group["title_for_plot"].notna().any() else feature
        resolved.append((dataset, feature, group, title))
    return resolved


def plot_ratio_improvement_distributions(
    ratio_df,
    *,
    plot_pairs=None,
    datasets=None,
    features=None,
    ncols=5,
    figsize_per_panel=(1.64, 1.75),
    output_path=None,
):
    resolved = resolve_ratio_plot_pairs(ratio_df, plot_pairs=plot_pairs, datasets=datasets, features=features)
    if not resolved:
        print("No dataset/feature pairs to plot.")
        return None

    nplots = len(resolved)
    nrows = int(np.ceil(nplots / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows))
    axes = np.atleast_1d(axes).flatten()

    for i, (dataset, feature, group, title) in enumerate(resolved):
        ax = axes[i]
        statistic = (group["ratio"].astype(float) - 1) * 100
        counts, bins, _ = ax.hist(statistic, bins=20, color="skyblue", alpha=0.7, density=True, zorder=3)

        xmin, xmax = bins[0], bins[-1]
        if len(statistic.dropna()) > 1 and statistic.nunique(dropna=True) > 1:
            x = np.linspace(xmin, xmax, 500)
            kde = stats.gaussian_kde(statistic.dropna(), bw_method=0.4)
            y = kde(x)
            ax.plot(x, y, color="orange", linewidth=1, zorder=2)
            ax.fill_between(x, y, color="orange", alpha=0.25, zorder=2)

        ax.axvline(np.mean(statistic), color="green", linestyle="--", label=f"Mean = {np.mean(statistic):.2f}")
        ax.axvline(0, color="red", linestyle="--", lw=1, alpha=0.6)
        ax.set_title(f"{display_dataset(dataset)}\n{title}", fontsize=8)
        ax.set_ylabel("Density" if i % ncols == 0 else "", fontsize=9)
        ax.set_xlabel("Improvement (%)" if i >= (nrows - 1) * ncols else "", fontsize=9)
        ax.set_xlim(xmin, xmax)
        ax.grid(True, which="major", linestyle="--", linewidth=0.25, alpha=0.7)
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)
        ax.tick_params(axis="both", labelsize=8)

    for j in range(len(resolved), len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout()
    save_fig(fig, output_path)
    return fig


In [ ]:
# All dataset x feature pairs available in ratio_df_all.
plot_ratio_improvement_distributions(
    ratio_df_all,
    ncols=5,
    output_path=OUT_DIR / "figure_ohe_vs_llm" / "improvement_plot.svg",
)
plt.show()

# Example: specific dataset-feature pairs and custom layout.
example_pairs = [("nmt", "correlation"), ("pard3", "correlation"), ("lov", "correlation"), ("gfp", "roc")]
plot_ratio_improvement_distributions(
    ratio_df_all,
    plot_pairs=example_pairs,
    ncols=2,
    output_path=OUT_DIR / "figure_ohe_vs_llm" / "improvement_plot_selected.svg",
)
plt.show()

# Example: one figure per dataset, all available features for that dataset.
for dataset in ratio_df_all["dataset"].dropna().unique():
    plot_ratio_improvement_distributions(
        ratio_df_all,
        datasets=[dataset],
        ncols=3,
        output_path=OUT_DIR / "figure_ohe_vs_llm" / str(dataset) / "improvement_plot.svg",
    )
    plt.show()
